In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

# =========================================================
# USER SETTINGS
# =========================================================

EDIN_CSV_DIR = Path("/Users/liuzimo/Desktop/MC data/ODE_workflow_outputs/csv")
GLAS_CSV_DIR = Path("/Users/liuzimo/Desktop/MC data/ODE_workflow_outputs_glasgow/csv")

OUTPUT_DIR = Path("/Users/liuzimo/Desktop/MC data/dissertation_tables")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

ROUND_ORDER = ["round1", "round2", "round3", "round4"]
ROUND_LABELS = {
    "round1": "Round 1",
    "round2": "Round 2",
    "round3": "Round 3",
    "round4": "Round 4",
}

FIT_MODE_LABELS = {
    "full_5_parameter_fit": "Full 5-parameter fit",
    "beta_only_fit_from_round1_baseline": "Beta-only fit from Round 1 baseline",
    "predicted_beta_from_round1_to_round3_trend": "Predicted beta from Round 1-3 trend",
}

# =========================================================
# HELPER FUNCTIONS
# =========================================================

def read_csv_checked(path):
    if not path.exists():
        raise FileNotFoundError(f"Missing file: {path}")
    return pd.read_csv(path)

def round_label(x):
    return ROUND_LABELS.get(str(x), str(x))

def sort_by_round(df, round_col="round"):
    if round_col in df.columns:
        df = df.copy()
        df["_round_order"] = pd.Categorical(df[round_col], categories=ROUND_ORDER, ordered=True)
        df = df.sort_values("_round_order").drop(columns="_round_order")
    return df

def coalesce_columns(df, candidates):
    out = pd.Series(np.nan, index=df.index, dtype=float)
    for col in candidates:
        if col in df.columns:
            vals = pd.to_numeric(df[col], errors="coerce")
            out = out.fillna(vals)
    return out

def yes_no(val):
    if pd.isna(val):
        return ""
    return "Yes" if bool(val) else "No"

def as_bool_series(series):
    return (
        series.astype(str)
        .str.strip()
        .str.lower()
        .map({"true": True, "false": False, "1": True, "0": False, "yes": True, "no": False})
    )

def format_p(p):
    if pd.isna(p):
        return "NA"
    if p < 1e-4:
        return "< 1e-4"
    return f"{p:.4f}"

def save_table(df, filename):
    path = OUTPUT_DIR / filename
    df.to_csv(path, index=False)
    print(f"Saved: {path}")

# =========================================================
# TABLE 1: EXPERIMENTAL DESIGN
# =========================================================

def build_table_1():
    rows = [
        ["Edinburgh", "Round 1", 0.005, "Normal + supplementary", 10, "Full fit"],
        ["Edinburgh", "Round 2", 0.010, "Normal + supplementary", 10, "Beta-only fit"],
        ["Edinburgh", "Round 3", 0.015, "Normal + supplementary", 10, "Beta-only fit"],
        ["Edinburgh", "Round 4", 0.020, "Normal only", 6, "Prediction test + beta-only fit"],
        ["Glasgow",   "Round 1", 0.005, "Normal only", 5, "Full fit"],
        ["Glasgow",   "Round 2", 0.010, "Normal only", 5, "Beta-only fit"],
        ["Glasgow",   "Round 3", 0.015, "Normal only", 5, "Beta-only fit"],
        ["Glasgow",   "Round 4", 0.020, "Normal only", 5, "Prediction test + beta-only fit"],
    ]
    return pd.DataFrame(rows, columns=[
        "Map",
        "Round",
        "Minecraft exposure setting",
        "Run type(s)",
        "Number of runs",
        "Role in analysis"
    ])

# =========================================================
# TABLE 2: SUMMARY MEASURES AND STATISTICAL TESTS
# =========================================================

def build_table_2():
    rows = [
        ["AUC of infected", "Summary metric", "Area under infected trajectory", "Cumulative infectious burden"],
        ["AUC of exposed + infected", "Summary metric", "Area under exposed plus infected trajectory", "Total active epidemic burden"],
        ["Peak infected", "Summary metric", "Maximum infected count", "Outbreak intensity"],
        ["Time to peak infected", "Summary metric", "Time at maximum infected count", "Outbreak timing"],
        ["RMSE", "Trajectory metric", "Root mean square error between ODE and Minecraft trajectory", "Trajectory agreement"],
        ["MAE", "Trajectory metric", "Mean absolute error between ODE and Minecraft trajectory", "Trajectory agreement"],
        ["Pearson correlation", "Trajectory metric", "Linear correlation between ODE and Minecraft trajectory", "Shape similarity"],
        ["Mann-Whitney U test", "Statistical test", "Two-group non-parametric comparison", "Normal vs supplementary and pairwise round comparisons"],
        ["Kruskal-Wallis test", "Statistical test", "Multi-group non-parametric comparison", "Overall round-to-round comparison"],
    ]
    return pd.DataFrame(rows, columns=[
        "Measure / test",
        "Type",
        "Definition",
        "Use in dissertation"
    ])

# =========================================================
# TABLE 3 / 7: ROUND-LEVEL MC AUC SUMMARY
# =========================================================

def build_round_auc_summary_table(csv_dir, map_name):
    df = read_csv_checked(csv_dir / "05_round_auc_summary.csv")
    df = sort_by_round(df, "round")
    df["Round"] = df["round"].map(round_label)
    cols = [
        "Round",
        "exposure",
        "n_runs",
        "mean_auc_infected",
        "median_auc_infected",
        "mean_auc_exposed_infected",
        "median_auc_exposed_infected",
        "mean_peak_infected",
        "median_peak_infected",
    ]
    df = df[cols].copy()
    df.columns = [
        "Round",
        "Exposure",
        "Number of runs",
        "Mean infected AUC",
        "Median infected AUC",
        "Mean exposed + infected AUC",
        "Median exposed + infected AUC",
        "Mean peak infected",
        "Median peak infected",
    ]
    df.insert(0, "Map", map_name)
    return df.round(4)

# =========================================================
# TABLE 4: FITTED ODE PARAMETERS
# =========================================================

def build_fitted_param_table_for_map(csv_dir, map_name):
    fit_main = read_csv_checked(csv_dir / "15_all_fit_tables_combined.csv")

    # Round 4 fit is stored separately
    round4_path = csv_dir / "20_round4_beta_only_fit_parameters.csv"
    if round4_path.exists():
        fit_r4 = read_csv_checked(round4_path)
        fit_all = pd.concat([fit_main, fit_r4], ignore_index=True, sort=False)
    else:
        fit_all = fit_main.copy()

    fit_all = sort_by_round(fit_all, "round")
    out = pd.DataFrame()
    out["Map"] = map_name
    out["Round"] = fit_all["round"].map(round_label)
    out["Fit mode"] = fit_all["fit_mode"].map(FIT_MODE_LABELS).fillna(fit_all["fit_mode"])
    out["Success"] = fit_all["success"].map(yes_no) if "success" in fit_all.columns else ""
    out["Cost"] = pd.to_numeric(fit_all.get("cost"), errors="coerce")

    out["Beta"] = pd.to_numeric(fit_all.get("beta"), errors="coerce")
    out["Epsilon"] = coalesce_columns(fit_all, ["epsilon", "epsilon_fixed"])
    out["Gamma"] = coalesce_columns(fit_all, ["gamma", "gamma_fixed"])
    out["Rho"] = coalesce_columns(fit_all, ["rho", "rho_fixed"])
    out["Alpha"] = coalesce_columns(fit_all, ["alpha", "alpha_fixed"])

    return out.round(6)

def build_table_4():
    edin = build_fitted_param_table_for_map(EDIN_CSV_DIR, "Edinburgh")
    glas = build_fitted_param_table_for_map(GLAS_CSV_DIR, "Glasgow")
    return pd.concat([edin, glas], ignore_index=True)

# =========================================================
# TABLE 5 / 8: ROUND 4 PREDICTED VS FITTED BETA
# =========================================================

def build_round4_beta_table(csv_dir, map_name):
    df = read_csv_checked(csv_dir / "27_round4_predicted_vs_fitted_beta.csv")
    out = pd.DataFrame()
    out["Map"] = map_name
    out["Round"] = df["round"].map(round_label)
    out["Exposure"] = pd.to_numeric(df["exposure"], errors="coerce")
    out["Predicted beta"] = pd.to_numeric(df["predicted_beta"], errors="coerce")
    out["Fitted beta"] = pd.to_numeric(df["fitted_beta"], errors="coerce")
    out["Absolute error"] = pd.to_numeric(df["absolute_error"], errors="coerce")
    out["Relative error (%)"] = pd.to_numeric(df["relative_error_percent"], errors="coerce")
    return out.round(6)

# =========================================================
# TABLE 6: EDINBURGH ODE VS MC INFECTED AUC
# =========================================================

def build_auc_distribution_table(csv_dir, map_name):
    df = read_csv_checked(csv_dir / "28_ode_vs_mc_auc_distribution_summary.csv")
    df = sort_by_round(df, "round")

    iqr_bool = as_bool_series(df["ode_within_mc_iqr"]) if "ode_within_mc_iqr" in df.columns else pd.Series([np.nan] * len(df))
    range_bool = as_bool_series(df["ode_within_mc_range"]) if "ode_within_mc_range" in df.columns else pd.Series([np.nan] * len(df))

    out = pd.DataFrame()
    out["Map"] = map_name
    out["Round"] = df["round"].map(round_label)
    out["Exposure"] = pd.to_numeric(df["exposure"], errors="coerce")
    out["Minecraft median infected AUC"] = pd.to_numeric(df["mc_auc_median"], errors="coerce")
    out["ODE infected AUC"] = pd.to_numeric(df["ode_auc_infected"], errors="coerce")
    out["ODE - MC median"] = pd.to_numeric(df["ode_minus_mc_median"], errors="coerce")
    out["Absolute difference"] = pd.to_numeric(df["ode_abs_diff_from_mc_median"], errors="coerce")
    out["ODE within MC IQR"] = iqr_bool.map(yes_no)
    out["ODE within MC full range"] = range_bool.map(yes_no)
    return out.round(4)

# =========================================================
# TABLE 9: EDINBURGH VS GLASGOW COMPARISON
# =========================================================

def get_overall_pvalue(tests_df, metric):
    row = tests_df[
        (tests_df["comparison_type"] == "overall_kruskal") &
        (tests_df["metric"] == metric)
    ]
    if len(row) == 0:
        return np.nan
    return pd.to_numeric(row.iloc[0]["p_value"], errors="coerce")

def count_true(series):
    b = as_bool_series(series)
    return int(b.fillna(False).sum())

def build_table_9():
    edin_tests = read_csv_checked(EDIN_CSV_DIR / "06_round_auc_tests.csv")
    glas_tests = read_csv_checked(GLAS_CSV_DIR / "06_round_auc_tests.csv")

    edin_r4 = read_csv_checked(EDIN_CSV_DIR / "27_round4_predicted_vs_fitted_beta.csv")
    glas_r4 = read_csv_checked(GLAS_CSV_DIR / "27_round4_predicted_vs_fitted_beta.csv")

    edin_auc = read_csv_checked(EDIN_CSV_DIR / "28_ode_vs_mc_auc_distribution_summary.csv")
    glas_auc = read_csv_checked(GLAS_CSV_DIR / "28_ode_vs_mc_auc_distribution_summary.csv")

    edin_beta = build_fitted_param_table_for_map(EDIN_CSV_DIR, "Edinburgh")
    glas_beta = build_fitted_param_table_for_map(GLAS_CSV_DIR, "Glasgow")

    def beta_increases(df):
        sub = df[df["Round"].isin(["Round 1", "Round 2", "Round 3"])].copy()
        sub = sub.sort_values("Round")
        vals = sub["Beta"].to_list()
        if len(vals) < 3:
            return ""
        return "Yes" if (vals[0] < vals[1] < vals[2]) else "No"

    edir4err = pd.to_numeric(edin_r4.iloc[0]["relative_error_percent"], errors="coerce")
    glasr4err = pd.to_numeric(glas_r4.iloc[0]["relative_error_percent"], errors="coerce")

    edin_all_below_median = (
        pd.to_numeric(edin_auc["ode_minus_mc_median"], errors="coerce") < 0
    ).all()
    glas_all_below_median = (
        pd.to_numeric(glas_auc["ode_minus_mc_median"], errors="coerce") < 0
    ).all()

    edin_iqr_any = count_true(edin_auc["ode_within_mc_iqr"]) > 0
    glas_iqr_any = count_true(glas_auc["ode_within_mc_iqr"]) > 0

    edin_range_count = count_true(edin_auc["ode_within_mc_range"])
    glas_range_count = count_true(glas_auc["ode_within_mc_range"])

    rows = [
        ["Runs per round", "10 pooled in Rounds 1-3; 6 in Round 4", "5 in each round"],
        ["Supplementary runs included", "Yes", "No"],
        ["Overall infected AUC round effect", f"p = {format_p(get_overall_pvalue(edin_tests, 'auc_infected'))}", f"p = {format_p(get_overall_pvalue(glas_tests, 'auc_infected'))}"],
        ["Overall exposed + infected AUC round effect", f"p = {format_p(get_overall_pvalue(edin_tests, 'auc_exposed_infected'))}", f"p = {format_p(get_overall_pvalue(glas_tests, 'auc_exposed_infected'))}"],
        ["Overall peak infected round effect", f"p = {format_p(get_overall_pvalue(edin_tests, 'peak_infected'))}", f"p = {format_p(get_overall_pvalue(glas_tests, 'peak_infected'))}"],
        ["Fitted beta increases from Round 1 to Round 3", beta_increases(edin_beta), beta_increases(glas_beta)],
        ["Round 4 beta relative error (%)", f"{edir4err:.2f}", f"{glasr4err:.2f}"],
        ["ODE infected AUC below MC median in all rounds", yes_no(edin_all_below_median), yes_no(glas_all_below_median)],
        ["ODE infected AUC within MC IQR in any round", yes_no(edin_iqr_any), yes_no(glas_iqr_any)],
        ["ODE infected AUC within MC full range (round count)", f"{edin_range_count}/{len(edin_auc)}", f"{glas_range_count}/{len(glas_auc)}"],
    ]

    return pd.DataFrame(rows, columns=["Feature", "Edinburgh", "Glasgow"])

# =========================================================
# MAIN
# =========================================================

def main():
    tables = {}

    # Manual tables
    tables["Table_1_Experimental_design"] = build_table_1()
    tables["Table_2_Summary_measures_and_tests"] = build_table_2()

    # CSV-derived tables
    tables["Table_3_Edinburgh_round_level_MC_AUC_summary"] = build_round_auc_summary_table(EDIN_CSV_DIR, "Edinburgh")
    tables["Table_4_Fitted_ODE_parameters"] = build_table_4()
    tables["Table_5_Edinburgh_Round4_predicted_vs_fitted_beta"] = build_round4_beta_table(EDIN_CSV_DIR, "Edinburgh")
    tables["Table_6_Edinburgh_ODE_vs_MC_infected_AUC"] = build_auc_distribution_table(EDIN_CSV_DIR, "Edinburgh")
    tables["Table_7_Glasgow_round_level_MC_AUC_summary"] = build_round_auc_summary_table(GLAS_CSV_DIR, "Glasgow")
    tables["Table_8_Glasgow_Round4_predicted_vs_fitted_beta"] = build_round4_beta_table(GLAS_CSV_DIR, "Glasgow")
    tables["Table_9_Edinburgh_vs_Glasgow_key_comparison"] = build_table_9()

    # Save each table as CSV
    for name, df in tables.items():
        save_table(df, f"{name}.csv")

    # Save all tables to one Excel workbook if openpyxl is available
    xlsx_path = OUTPUT_DIR / "Dissertation_Tables.xlsx"
    try:
        with pd.ExcelWriter(xlsx_path, engine="openpyxl") as writer:
            for name, df in tables.items():
                sheet_name = name[:31]  # Excel sheet name limit
                df.to_excel(writer, sheet_name=sheet_name, index=False)
        print(f"Saved: {xlsx_path}")
    except ImportError:
        print("openpyxl not installed, so Excel workbook was not written.")

    # Optional: print previews
    print("\nPreview of generated tables:\n")
    for name, df in tables.items():
        print("=" * 80)
        print(name)
        print("=" * 80)
        print(df.head(20).to_string(index=False))
        print()

if __name__ == "__main__":
    main()


Saved: /Users/liuzimo/Desktop/MC data/dissertation_tables/Table_1_Experimental_design.csv
Saved: /Users/liuzimo/Desktop/MC data/dissertation_tables/Table_2_Summary_measures_and_tests.csv
Saved: /Users/liuzimo/Desktop/MC data/dissertation_tables/Table_3_Edinburgh_round_level_MC_AUC_summary.csv
Saved: /Users/liuzimo/Desktop/MC data/dissertation_tables/Table_4_Fitted_ODE_parameters.csv
Saved: /Users/liuzimo/Desktop/MC data/dissertation_tables/Table_5_Edinburgh_Round4_predicted_vs_fitted_beta.csv
Saved: /Users/liuzimo/Desktop/MC data/dissertation_tables/Table_6_Edinburgh_ODE_vs_MC_infected_AUC.csv
Saved: /Users/liuzimo/Desktop/MC data/dissertation_tables/Table_7_Glasgow_round_level_MC_AUC_summary.csv
Saved: /Users/liuzimo/Desktop/MC data/dissertation_tables/Table_8_Glasgow_Round4_predicted_vs_fitted_beta.csv
Saved: /Users/liuzimo/Desktop/MC data/dissertation_tables/Table_9_Edinburgh_vs_Glasgow_key_comparison.csv
Saved: /Users/liuzimo/Desktop/MC data/dissertation_tables/Dissertation_Tables.